# Personalized Career Path Recommendation with AI  
## Notebook 1: Data Loading and Preparation

This notebook prepares the starter dataset for the career recommendation system.

The main goal of this step is to load the role dataset, repair the file structure if needed, clean the skill-related columns, and transform the data into a format that can later be used for recommendation scoring.

By the end of this notebook, the dataset will be ready for role matching, skill-gap analysis, and recommendation generation.

In [12]:
import pandas as pd
import csv

In [13]:
raw = pd.read_excel("data/career_roles_seed.xlsx")
raw.head()

,"role_title,domain,role_description,required_skills,preferred_skills,experience_level,education_level,interest_tags"
0,"Data Analyst,Data & Analytics,""Analyses struct..."
1,"Business Analyst,Business & Analytics,""Works w..."
2,"BI Analyst,Data & Analytics,""Builds dashboards..."
3,"Product Analyst,Data & Analytics,""Uses product..."
4,"Operations Analyst,Operations & Analytics,""Ana..."


## Inspect the raw structure

This step helps confirm whether the workbook loaded correctly.

If the dataset is in the correct format, it should contain multiple columns such as:
- role title
- domain
- role description
- required skills
- preferred skills
- experience level
- education level
- interest tags

If it instead loads as a single column, we will rebuild it into a proper table.

In [14]:
print(raw.shape)
print(raw.columns.tolist())

(20, 1)
['role_title,domain,role_description,required_skills,preferred_skills,experience_level,education_level,interest_tags']


## Repair the workbook structure

The uploaded workbook currently contains CSV-style rows inside a single Excel column.

To fix this, we:
1. collect the header and row text
2. parse the text using CSV rules
3. rebuild the dataset into a clean DataFrame
4. save the repaired version for later use

This ensures that each field is placed in the correct column.

In [15]:
rows = [raw.columns[0]] + raw.iloc[:, 0].dropna().astype(str).tolist()
parsed = list(csv.reader(rows))

df = pd.DataFrame(parsed[1:], columns=parsed[0])
df.to_excel("career_roles_seed_fixed.xlsx", index=False)

print(df.shape)
print(df.columns.tolist())
df.head()

(20, 8)
['role_title', 'domain', 'role_description', 'required_skills', 'preferred_skills', 'experience_level', 'education_level', 'interest_tags']


,role_title,domain,role_description,required_skills,preferred_skills,experience_level,education_level,interest_tags
0,Data Analyst,Data & Analytics,"Analyses structured data to produce reports, d...",SQL|Excel|Python|Data Visualization|Statistics,Power BI|Tableau|Communication,Entry,Bachelor/MSc,Analytics|Business|Problem Solving
1,Business Analyst,Business & Analytics,Works with stakeholders to understand business...,Excel|Business Analysis|Communication|Requirem...,SQL|Power BI|Project Management,Entry,Bachelor/MSc,Business|Strategy|Problem Solving
2,BI Analyst,Data & Analytics,Builds dashboards and reporting solutions to t...,SQL|Power BI|Excel|Data Visualization|Dashboar...,Tableau|Python|Business Analysis,Entry,Bachelor/MSc,Reporting|Analytics|Business
3,Product Analyst,Data & Analytics,Uses product and user data to evaluate feature...,SQL|Python|A/B Testing|Statistics|Data Visuali...,Product Thinking|Communication|Dashboarding,Entry,Bachelor/MSc,Product|Experimentation|Analytics
4,Operations Analyst,Operations & Analytics,Analyses operational data to improve workflows...,Excel|SQL|Data Cleaning|Reporting|Problem Solving,Python|Dashboarding|Communication,Entry,Bachelor/MSc,Operations|Efficiency|Analytics


## Understand the cleaned dataset

Now that the structure has been repaired, the dataset should contain one row per career role.

Each row represents a possible career path, and each column stores information that will later be used for recommendation logic.

The most important columns for the recommendation engine are:
- `role_title`
- `domain`
- `role_description`
- `required_skills`
- `preferred_skills`
- `experience_level`
- `education_level`
- `interest_tags`

In [16]:
df.info()
df.head(3)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   role_title        20 non-null     object
 1   domain            20 non-null     object
 2   role_description  20 non-null     object
 3   required_skills   20 non-null     object
 4   preferred_skills  20 non-null     object
 5   experience_level  20 non-null     object
 6   education_level   20 non-null     object
 7   interest_tags     20 non-null     object
dtypes: object(8)
memory usage: 1.4+ KB


,role_title,domain,role_description,required_skills,preferred_skills,experience_level,education_level,interest_tags
0,Data Analyst,Data & Analytics,"Analyses structured data to produce reports, d...",SQL|Excel|Python|Data Visualization|Statistics,Power BI|Tableau|Communication,Entry,Bachelor/MSc,Analytics|Business|Problem Solving
1,Business Analyst,Business & Analytics,Works with stakeholders to understand business...,Excel|Business Analysis|Communication|Requirem...,SQL|Power BI|Project Management,Entry,Bachelor/MSc,Business|Strategy|Problem Solving
2,BI Analyst,Data & Analytics,Builds dashboards and reporting solutions to t...,SQL|Power BI|Excel|Data Visualization|Dashboar...,Tableau|Python|Business Analysis,Entry,Bachelor/MSc,Reporting|Analytics|Business


## Convert skill and interest fields into lists

The skill and interest columns are currently stored as text separated by the `|` symbol.

For example:

`SQL|Python|Statistics`

This format is useful for saving the dataset, but not ideal for analysis.

In this step, the text is split into Python lists so each role has:
- a list of required skills
- a list of preferred skills
- a list of interest tags

This will make it much easier to compare a user's profile against each role later.

In [17]:
list_cols = ["required_skills", "preferred_skills", "interest_tags"]

for col in list_cols:
    df[col] = df[col].fillna("").apply(
        lambda x: [item.strip() for item in str(x).split("|") if item.strip()]
    )

df.head(3)

,role_title,domain,role_description,required_skills,preferred_skills,experience_level,education_level,interest_tags
0,Data Analyst,Data & Analytics,"Analyses structured data to produce reports, d...","[SQL, Excel, Python, Data Visualization, Stati...","[Power BI, Tableau, Communication]",Entry,Bachelor/MSc,"[Analytics, Business, Problem Solving]"
1,Business Analyst,Business & Analytics,Works with stakeholders to understand business...,"[Excel, Business Analysis, Communication, Requ...","[SQL, Power BI, Project Management]",Entry,Bachelor/MSc,"[Business, Strategy, Problem Solving]"
2,BI Analyst,Data & Analytics,Builds dashboards and reporting solutions to t...,"[SQL, Power BI, Excel, Data Visualization, Das...","[Tableau, Python, Business Analysis]",Entry,Bachelor/MSc,"[Reporting, Analytics, Business]"


##  Check one sample row

This step helps verify that the skill-related columns were converted correctly.

Instead of one long string, the values should now appear as proper Python lists.

This is important because the recommendation system will later calculate:
- skill overlap
- matched skills
- missing skills
- interest alignment

In [18]:
df.loc[0]

role_title                                               Data Analyst
domain                                               Data & Analytics
role_description    Analyses structured data to produce reports, d...
required_skills     [SQL, Excel, Python, Data Visualization, Stati...
preferred_skills                   [Power BI, Tableau, Communication]
experience_level                                                Entry
education_level                                          Bachelor/MSc
interest_tags                  [Analytics, Business, Problem Solving]
Name: 0, dtype: object

## Create a sample user profile

To test the recommendation pipeline, we create a sample user profile.

This acts as an example input for the system and includes:
- education level
- experience level
- current skills
- interests

Later, this profile will be replaced by real user input from a Streamlit app or form.

In [19]:
user_profile = {
    "education_level": "MSc",
    "experience_level": "Entry",
    "current_skills": ["Python", "SQL", "Statistics", "Data Visualization"],
    "interests": ["AI", "Analytics", "Problem Solving"]
}

user_profile

{'education_level': 'MSc',
 'experience_level': 'Entry',
 'current_skills': ['Python', 'SQL', 'Statistics', 'Data Visualization'],
 'interests': ['AI', 'Analytics', 'Problem Solving']}

## Define helper functions for scoring

The recommendation system needs a way to compare the user profile with each role.

The helper functions below do the following:

- `overlap_score()` measures how much overlap exists between two lists
- `experience_score()` compares the user's experience level with the role level
- `education_score()` compares the user's education level with the role requirement

These functions provide the foundation for the first version of the recommendation engine.

In [20]:
def overlap_score(user_items, role_items):
    user_set = set([x.strip().lower() for x in user_items])
    role_set = set([x.strip().lower() for x in role_items])

    if len(role_set) == 0:
        return 0.0

    return len(user_set & role_set) / len(role_set)


def experience_score(user_exp, role_exp):
    user_exp = str(user_exp).strip().lower()
    role_exp = str(role_exp).strip().lower()

    if user_exp in role_exp:
        return 1.0
    elif user_exp == "entry" and role_exp == "junior":
        return 0.7
    elif user_exp == "entry" and role_exp == "entry/junior":
        return 1.0
    else:
        return 0.4


def education_score(user_edu, role_edu):
    user_edu = str(user_edu).strip().lower()
    role_edu = str(role_edu).strip().lower()

    if user_edu in role_edu:
        return 1.0
    else:
        return 0.6

## Compute the first recommendation scores

In this step, each role is scored against the sample user profile.

The scoring is broken into several parts:
- required skill match
- preferred skill match
- interest alignment
- experience alignment
- education alignment

These separate scores will later be combined into one final career fit score.

In [21]:
df["required_skill_score"] = df["required_skills"].apply(
    lambda x: overlap_score(user_profile["current_skills"], x)
)

df["preferred_skill_score"] = df["preferred_skills"].apply(
    lambda x: overlap_score(user_profile["current_skills"], x)
)

df["interest_score"] = df["interest_tags"].apply(
    lambda x: overlap_score(user_profile["interests"], x)
)

df["experience_score"] = df["experience_level"].apply(
    lambda x: experience_score(user_profile["experience_level"], x)
)

df["education_score"] = df["education_level"].apply(
    lambda x: education_score(user_profile["education_level"], x)
)

## Create a final fit score

The final fit score combines the individual components into one ranking metric.

The current weighting gives the strongest importance to required skills, because they are the most direct indicator of role suitability.

This is a simple and explainable first version of the recommendation formula.  
It can be improved later by tuning weights or adding NLP-based similarity.

In [22]:
df["fit_score"] = (
    0.45 * df["required_skill_score"] +
    0.15 * df["preferred_skill_score"] +
    0.20 * df["interest_score"] +
    0.10 * df["experience_score"] +
    0.10 * df["education_score"]
)

## Rank the career roles

Now that each role has a fit score, we sort the dataset from highest to lowest.

This gives the first version of the recommendation output:
the top-ranked roles are the careers that best match the current user profile based on the scoring logic used above.

In [23]:
ranked_df = df.sort_values(by="fit_score", ascending=False).reset_index(drop=True)

ranked_df[[
    "role_title",
    "domain",
    "fit_score",
    "required_skill_score",
    "preferred_skill_score",
    "interest_score"
]].head(10)

,role_title,domain,fit_score,required_skill_score,preferred_skill_score,interest_score
0,Data Analyst,Data & Analytics,0.693333,0.8,0.000000,0.666667
1,Data Scientist,AI & Analytics,0.693333,0.8,0.000000,0.666667
2,Product Analyst,Data & Analytics,0.626667,0.8,0.000000,0.333333
3,Decision Scientist,AI & Analytics,0.556667,0.6,0.333333,0.333333
4,Marketing Analyst,Marketing & Analytics,0.546667,0.4,0.666667,0.333333
5,BI Analyst,Data & Analytics,0.496667,0.4,0.333333,0.333333
6,Customer Insights Analyst,Business & Analytics,0.496667,0.4,0.333333,0.333333
7,Fraud Analyst,Risk & Analytics,0.430000,0.4,0.333333,0.000000
8,Risk Analyst,Finance & Analytics,0.430000,0.4,0.333333,0.000000
9,Machine Learning Engineer,AI & ML,0.416667,0.4,0.000000,0.333333


## Identify missing required skills

A good recommendation system should not only suggest roles, but also explain what the user is missing.

This step identifies required skills that are not yet present in the user's current profile.

These missing skills will later be used for:
- skill-gap analysis
- learning recommendations
- roadmap generation

In [24]:
def missing_skills(user_skills, required_skills):
    user_set = set([x.strip().lower() for x in user_skills])
    missing = [skill for skill in required_skills if skill.strip().lower() not in user_set]
    return missing

ranked_df["missing_required_skills"] = ranked_df["required_skills"].apply(
    lambda x: missing_skills(user_profile["current_skills"], x)
)

ranked_df[["role_title", "fit_score", "missing_required_skills"]].head(5)

,role_title,fit_score,missing_required_skills
0,Data Analyst,0.693333,[Excel]
1,Data Scientist,0.693333,[Machine Learning]
2,Product Analyst,0.626667,[A/B Testing]
3,Decision Scientist,0.556667,"[Experimentation, Communication]"
4,Marketing Analyst,0.546667,"[Excel, Reporting, Communication]"


## Identify matched required skills

In addition to missing skills, it is also useful to show which required skills the user already has.

This improves explainability and helps justify why a role was recommended.

In [25]:
def matched_skills(user_skills, required_skills):
    user_set = set([x.strip().lower() for x in user_skills])
    return [skill for skill in required_skills if skill.strip().lower() in user_set]

ranked_df["matched_required_skills"] = ranked_df["required_skills"].apply(
    lambda x: matched_skills(user_profile["current_skills"], x)
)

ranked_df[["role_title", "matched_required_skills", "missing_required_skills"]].head(5)

,role_title,matched_required_skills,missing_required_skills
0,Data Analyst,"[SQL, Python, Data Visualization, Statistics]",[Excel]
1,Data Scientist,"[Python, SQL, Statistics, Data Visualization]",[Machine Learning]
2,Product Analyst,"[SQL, Python, Statistics, Data Visualization]",[A/B Testing]
3,Decision Scientist,"[Python, SQL, Statistics]","[Experimentation, Communication]"
4,Marketing Analyst,"[Data Visualization, Statistics]","[Excel, Reporting, Communication]"


## Save the prepared output

The ranked results are saved so they can be reused in later notebooks.

This makes the workflow more modular and allows the next stage of the project to focus on:
- deeper skill-gap analysis
- roadmap generation
- user-facing recommendation summaries

In [26]:
ranked_df.to_excel("career_recommendation_results.xlsx", index=False)
print("Saved as career_recommendation_results.xlsx")

Saved as career_recommendation_results.xlsx


## Summary

In this notebook, the dataset was:
- loaded from Excel
- repaired into a usable tabular format
- cleaned and transformed
- converted into list-based skill fields
- matched against a sample user profile
- scored and ranked to produce the first recommendation output

The next notebook will focus on improving recommendation quality and building a clearer skill-gap analysis.